# Task B batch #2 - warm start and class-weighted CE

**Baseline.** The plain 3-class A160 model is already a real Task B baseline:

| arm | mean best macro-F1 | 3-seed ensemble |
|---|---:|---:|
| `TB` from scratch, plain CE | **.5568** | **.5698** |

Per-class ensemble F1: `no .690`, `implicit .435`, `explicit .585`. The bottleneck
is the implicit class, followed by explicit recall. The model also stops early
(best epoch 3-5), which suggests the task is hard for the current initialization
rather than under-trained.

**This batch changes no architecture.** It tests three cheap levers with the same
A160 recipe:

| arm | initialization | loss |
|---|---|---|
| `CW` | random | class-weighted CE, `sqrt` weights |
| `CWI` | random | class-weighted CE, `inverse` weights |
| `WS` | Task A `A160r` checkpoint | plain CE |
| `WC` | Task A `A160r` checkpoint | class-weighted CE, `sqrt` weights |

`WS` and `WC` are skipped automatically if no Task A checkpoint is attached.

For `--class-weighting sqrt`, train counts `3351/1665/1873` become weights
`[.799, 1.133, 1.068]`. For `inverse`, they become `[.685, 1.379, 1.226]`.
Weights are computed from the train split only and normalized to mean 1.

**Decision rule.** Compare 3-seed mean and 3-seed ensemble against `TB` .5568/.5698.
A useful arm should clear .58 mean or .59 ensemble without collapsing the `no`
class. If none does, stop tuning this architecture and move to the Task B LLM.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '4d51dd9fc9ae8ecfdd0dd1bd1a0866dd4d9b3f01'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)

## 1. Data, GPU, and the optional Task A checkpoint

Attach `amnpham/lgbtqia` as before. To enable the warm-start arms, also attach a
Kaggle dataset containing the Task A `A160r` checkpoint (for example
`weights/best_xattn_stereotype.pt`). The notebook searches `/kaggle/input/**` for
`best_xattn_stereotype.pt` and `*A160*.pt`; set `A160_CKPT` explicitly if the file
has a different name.

Expected Task B val counts: `343 no / 175 implicit / 220 explicit`.

In [ ]:
import glob
import json
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
df_all = data_mod.load_frame(
    languages=['EN', 'IT', 'NL'],
    label_column='hate_speech',
    label_mode='hate3',
)
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
counts = np.bincount(labs, minlength=3)
assert len(labs) == 738, f'unexpected val size: {len(labs)}'
assert counts.tolist() == [343, 175, 220], f'unexpected Task B val counts: {counts.tolist()}'
np.save('weights/val_labels_taskb.npy', labs)
np.save('weights/val_langs_taskb.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos_taskb.npy', df_val['yt_title'].to_numpy(dtype=str))
np.save('weights/val_ids_taskb.npy', df_val['StereoQueerEval_id'].to_numpy(dtype=str))
print('val:', len(df_val), 'rows |', df_val['yt_title'].nunique(), 'videos |',
      dict(zip(('no', 'implicit', 'explicit'), counts.tolist())), flush=True)
print('train counts:',
      dict(zip(('no', 'implicit', 'explicit'),
               np.bincount(df_train['st_y'].astype(int), minlength=3).tolist())), flush=True)

# Optional warm-start checkpoint. A160r is a shared-encoder, 3-block, 2-way model.
env_ckpt = os.environ.get('A160_CKPT', '').strip()
candidates = []
if env_ckpt and os.path.exists(env_ckpt):
    candidates.append(env_ckpt)
for pattern in (
    '/kaggle/input/**/best_xattn_stereotype.pt',
    '/kaggle/input/**/*A160r*.pt',
    '/kaggle/input/**/*A160*.pt',
    '/kaggle/input/**/*stereotype*.pt',
):
    candidates.extend(sorted(glob.glob(pattern, recursive=True)))
candidates = list(dict.fromkeys(candidates))

INIT_CKPT = None
for c in candidates:
    try:
        sd = torch.load(c, map_location='cpu')
    except Exception as e:                       # noqa: BLE001
        print(f'!! cannot read candidate checkpoint {c}: {e}', flush=True)
        continue
    head = sd.get('head.2.weight')
    has_encoder_v = any(k.startswith('encoder_v') for k in sd)
    print(f'candidate: {c} | head {None if head is None else tuple(head.shape)} '
          f'| separate_encoder={has_encoder_v} | tensors={len(sd)}', flush=True)
    if head is not None and tuple(head.shape)[0] == 2 and not has_encoder_v and INIT_CKPT is None:
        INIT_CKPT = c
    del sd

if INIT_CKPT:
    print('warm-start checkpoint:', INIT_CKPT, flush=True)
else:
    print('no compatible A160r checkpoint found; WS/WC arms will be skipped', flush=True)

## 2. Run list and helpers

The architecture is identical to batch 1. `--class-weighting` computes weights from
the train split; `--init-from` discards the old 2-way Task A head and starts the
3-way Task B head fresh while transferring the encoders, decoder blocks and pool
query. All arms use split-seed 42 and seeds 42/43/44.

In [ ]:
BASE = dict(
    dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
    backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
    pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
    max_len=160, video_max_len=256, no_amp=False,
    backbone=model_xattn.DEFAULT_BACKBONE,
)

RUNS = []
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'CW_s{s}', seed=s, split_seed=42,
                     class_weighting='sqrt', init_from=None))
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'CWI_s{s}', seed=s, split_seed=42,
                     class_weighting='inverse', init_from=None))
if INIT_CKPT:
    for s in (42, 43, 44):
        RUNS.append(dict(BASE, tag=f'WS_s{s}', seed=s, split_seed=42,
                         class_weighting='none', init_from=INIT_CKPT))
    for s in (42, 43, 44):
        RUNS.append(dict(BASE, tag=f'WC_s{s}', seed=s, split_seed=42,
                         class_weighting='sqrt', init_from=INIT_CKPT))

BASE_RECORDED = dict(mean=0.5568, ensemble=0.5698)


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    probs = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(hist) and os.path.exists(out):
        print(f"-- skip (checkpoint + history exist): {r['tag']}", flush=True)
        return True
    init = f"--init-from {r['init_from']} " if r['init_from'] else ''
    cmd = (
        'python src/train_xattn.py --task b --languages EN IT NL '
        '--epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 --no-pooled-v --shared-encoder "
        f"--class-weighting {r['class_weighting']} {init}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist} --probs-out {probs}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {float(h.loc[bi, 'val_macro_f1']):.4f} | "
          f"acc {float(h.loc[bi, 'val_acc']):.4f} | {len(h)} epochs", flush=True)
    return True


def ensure_probs(r):
    p = f"weights/{r['tag']}_val_probs.npy"
    ckpt = f"weights/{r['tag']}.pt"
    if os.path.exists(p):
        return
    if not os.path.exists(ckpt):
        print(f'-- no checkpoint for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        num_classes=3,
    ).to(dev)
    net.load_state_dict(torch.load(ckpt, map_location=dev))
    net.eval()
    tok_r = load_tokenizer(r['backbone'])
    if tok_r.pad_token is None:
        tok_r.pad_token = tok_r.eos_token or tok_r.sep_token
    loader = tx.make_loader(df_val, tok_r, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(p, o['probs'])
    m = tx.metrics_from_taskb(o['preds'], o['labels'])
    print(f"   {r['tag']}: recovered probs | macro_f1 {m['macro_f1']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)

## 3. Train

Resume-safe. The class-weight arms run first; the warm-start arms are included only
if a compatible A160r checkpoint was found. Small results are zipped after every run.

In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} (elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        ensure_probs(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! ensure_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        zip_small()
    except Exception as e:                           # noqa: BLE001
        print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
print(f'all {len(RUNS)} runs finished in {(time.time() - t0) / 3600:.2f} h', flush=True)

## 4. Summary

Per-seed results, per-arm mean and 3-seed ensemble, per-class F1, and the deltas
against the batch-1 baseline. Raw per-seed probabilities are saved for the local
paired video-cluster bootstrap.

In [ ]:
from sklearn.metrics import f1_score

labs = np.load('weights/val_labels_taskb.npy')
langs = np.load('weights/val_langs_taskb.npy', allow_pickle=True)
names = ('no', 'implicit', 'explicit')
rows = []
probs_by_tag = {}

for r in RUNS:
    hist = f"weights/{r['tag']}_history.csv"
    p = f"weights/{r['tag']}_val_probs.npy"
    if not (os.path.exists(hist) and os.path.exists(p)):
        print(f'missing history/probs: {r["tag"]}', flush=True)
        continue
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    probs = np.load(p)
    pred = probs.argmax(1)
    m = tx.metrics_from_taskb(pred.tolist(), labs.tolist())
    probs_by_tag[r['tag']] = probs
    rows.append(dict(
        arm=r['tag'].rsplit('_s', 1)[0], tag=r['tag'], seed=r['seed'],
        weighting=r['class_weighting'], warm=bool(r['init_from']),
        best_epoch=int(h.loc[bi, 'epoch']),
        best_hist_f1=float(h.loc[bi, 'val_macro_f1']),
        reload_f1=float(m['macro_f1']), acc=float(m['acc']),
        f1_no=float(m['f1_no']), f1_implicit=float(m['f1_implicit']),
        f1_explicit=float(m['f1_explicit']),
        pred_no=float((pred == 0).mean()),
        pred_implicit=float((pred == 1).mean()),
        pred_explicit=float((pred == 2).mean()),
    ))
    print(f"{r['tag']}: ep {int(h.loc[bi, 'epoch']):>2} | "
          f"hist {float(h.loc[bi, 'val_macro_f1']):.4f} | reload {m['macro_f1']:.4f} | "
          f"F1 no {m['f1_no']:.3f} imp {m['f1_implicit']:.3f} exp {m['f1_explicit']:.3f}",
          flush=True)

summ = pd.DataFrame(rows).sort_values(['arm', 'seed'])
summ.to_csv('weights/taskB_batch2_summary.csv', index=False)
print()
print(summ.to_string(index=False), flush=True)

arm_rows = []
arm_ens = {}
print()
for arm in ('CW', 'CWI', 'WS', 'WC'):
    tags = [r['tag'] for r in RUNS if r['tag'].startswith(arm + '_s')]
    tags = [t for t in tags if t in probs_by_tag]
    if len(tags) != 3:
        print(f'{arm}: {len(tags)}/3 seeds; skipped', flush=True)
        continue
    arm_df = summ[summ.arm == arm]
    mean_best = float(arm_df['best_hist_f1'].mean())
    ens = np.mean([probs_by_tag[t] for t in tags], axis=0)
    ep = ens.argmax(1)
    em = tx.metrics_from_taskb(ep.tolist(), labs.tolist())
    arm_ens[arm] = ens
    np.save(f'weights/taskB_batch2_{arm}_ensemble_probs.npy', ens)
    arm_rows.append(dict(arm=arm, mean_best=mean_best, ensemble=float(em['macro_f1']),
                         acc=float(em['acc']), f1_no=float(em['f1_no']),
                         f1_implicit=float(em['f1_implicit']),
                         f1_explicit=float(em['f1_explicit'])))
    print(f"{arm:<4} mean best {mean_best:.4f} ({mean_best - BASE_RECORDED['mean']:+.4f}) | "
          f"ensemble {em['macro_f1']:.4f} ({em['macro_f1'] - BASE_RECORDED['ensemble']:+.4f}) | "
          f"F1 no {em['f1_no']:.3f} imp {em['f1_implicit']:.3f} exp {em['f1_explicit']:.3f}",
          flush=True)
    print(tx.format_confusion_matrix_taskb(labs, ep), flush=True)

arm_summ = pd.DataFrame(arm_rows)
arm_summ.to_csv('weights/taskB_batch2_arms.csv', index=False)
if len(arm_summ):
    best_arm = arm_summ.sort_values('ensemble', ascending=False).iloc[0]['arm']
    print(f'\nbest arm by ensemble: {best_arm}', flush=True)
    ep = arm_ens[best_arm].argmax(1)
    print('--- PER LANGUAGE (best arm ensemble) ---', flush=True)
    for lang in sorted(set(langs.tolist())):
        msk = langs == lang
        ml = tx.metrics_from_taskb(ep[msk].tolist(), labs[msk].tolist())
        print(f"  [{lang}] n {int(msk.sum())} | macro_f1 {ml['macro_f1']:.4f} | "
              f"F1 no {ml['f1_no']:.3f} imp {ml['f1_implicit']:.3f} exp {ml['f1_explicit']:.3f}",
              flush=True)

maj_f1 = f1_score(labs, np.zeros_like(labs), labels=[0, 1, 2], average='macro', zero_division=0)
print()
print(f'batch-1 baseline: mean {BASE_RECORDED["mean"]:.4f} | ensemble {BASE_RECORDED["ensemble"]:.4f}', flush=True)
print(f'majority {maj_f1:.4f} | TF-IDF .4912', flush=True)
if len(arm_summ) and float(arm_summ['ensemble'].max()) > BASE_RECORDED['ensemble'] + 0.005:
    print('decision: a new arm clears the baseline by >0.5pt; adopt it for the next screen', flush=True)
else:
    print('decision: no clear xattn win; move to Task B LLM / ordinal-hierarchical next', flush=True)
zip_small()
print('TASKB BATCH2 COMPLETE', flush=True)